# STAIR4-CSGC v4 — Training Pipeline trên Kaggle

Tham khảo cách tổ chức của notebook v3/v2, pipeline này gọi trực tiếp **main_stair4_v4.py**:
**cấu hình → môi trường → dữ liệu → preflight → graph audit → benchmark → train/resume → biểu đồ → báo cáo → đóng gói**.

V4 hiệu chỉnh graph item từ train interactions, giữ MI/FSC/BPR và scorer STAIR.
Không có contrastive loss, gate học, warmup hoặc pha lượng tử. Preprocessing/SVD vẫn có chi phí.
Mặc định chạy **Baby, seed 1, pilot 50 epochs, V4-B1 và V4-C**.
V4-B1 là baseline-recovery qua entry point v4, không phải một lần chạy main.py gốc.

Bật GPU và Internet nếu cần clone/cài dependencies; attach dataset trước Run All.
Code v4 phải được push lên GitHub hoặc upload đầy đủ. Không có metric hay biểu đồ mô phỏng điền sẵn.
Tài liệu: docs/giai_doan_4/STAIR4_v4_Report.md, STAIR4_v4_Implementation.md và STAIR4_v4_Source_Audit.md.


## 1. Bảng điều khiển

Chọn datasets, seeds và pilot/full ở đây. Để đường dẫn dữ liệu None để tự dò; nhiều kết quả khớp sẽ yêu cầu chỉ định đường dẫn.
Mặc định không chạy Sports/Electronics. Full dùng số epochs từ YAML baseline (500).
Đổi EXPERIMENT_ID cho kế hoạch mới; giữ output cũ để resume. Tuning chỉ dùng validation.


In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, shutil, subprocess, importlib.metadata, uuid

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO = WORK / 'STAIR-Enhanced' if Path('/kaggle/working').exists() else Path.cwd()
REPO_URL = 'https://github.com/ThanhChuong12/STAIR-Enhanced.git'
EXPECTED_COMMIT = None
INSTALL_DEPENDENCIES = True
DEVICE = '0'  # First GPU. Use 'cpu' only for local smoke/debug.
DATASETS_TO_RUN = ['baby']
DATASET_SOURCES = {'baby': None, 'sports': None, 'electronics': None}
SEEDS = [1]  # Use paired seeds, e.g. [1, 2, 3, 4, 5], for confirmation.
ARMS = ['V4-B1', 'V4-C']
RUN_MODE = 'pilot'  # pilot | full
PILOT_EPOCHS = 50
RUN_BENCHMARK = True
BENCHMARK_EPOCHS = 3
CORE_ALPHA = 0.25
EXPERIMENT_ID = time.strftime('v4_%Y%m%d_%H%M%S')
RESUME_RUN = None  # Absolute path to an existing run with command.json and checkpoints/.
RESUME_TARGET_EPOCHS = None
MAKE_ARCHIVE = True

DATASET_NAMES = {'baby': 'Amazon2014Baby_550_MMRec', 'sports': 'Amazon2014Sports_550_MMRec',
                 'electronics': 'Amazon2014Electronics_550_MMRec'}
DATA_ROOT = WORK / 'stair4_v4_data'
ARTIFACT_ROOT = WORK / 'stair4_v4_runs' / EXPERIMENT_ID
REPORT_DIR = ARTIFACT_ROOT / 'reports'
CACHE_ROOT = WORK / 'stair4_v4_cache'
INPUT_ROOT = Path('/kaggle/input')
REQUIRED_FILES = ('train.txt', 'valid.txt', 'test.txt', 'textual_modality.pkl', 'visual_modality.pkl')
PRESETS = {
    'V4-B1': {'alpha': 0.0}, 'V4-C': {'alpha': CORE_ALPHA},
    'V4-NS': {'alpha': CORE_ALPHA}, 'V4-ND': {'alpha': CORE_ALPHA},
    'V4-NA': {'alpha': CORE_ALPHA, 'activity_weighting': 'uniform'},
    'V4-SH': {'alpha': CORE_ALPHA, 'shuffle_seed': 1},
    'V4-SM': {'alpha': 0.0, 'identity_mix': 0.1},
}
assert RUN_MODE in {'pilot', 'full'} and DATASETS_TO_RUN
assert set(DATASETS_TO_RUN) <= set(DATASET_NAMES) and ARMS and set(ARMS) <= set(PRESETS)
assert SEEDS and all(isinstance(s, int) and s >= 0 for s in SEEDS)
assert len(SEEDS) == len(set(SEEDS)) and len(ARMS) == len(set(ARMS))
assert PILOT_EPOCHS > 0 and BENCHMARK_EPOCHS > 0 and 0 <= CORE_ALPHA <= 1
assert Path(EXPERIMENT_ID).name == EXPERIMENT_ID and EXPERIMENT_ID not in {'.', '..'}
for directory in (ARTIFACT_ROOT, REPORT_DIR, CACHE_ROOT, ARTIFACT_ROOT / 'preflight'):
    directory.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ, PYTHONUTF8='1', PYTHONUNBUFFERED='1', OMP_NUM_THREADS='2',
           MKL_NUM_THREADS='2', CUBLAS_WORKSPACE_CONFIG=':4096:8', MPLBACKEND='Agg')
print('Plan:', DATASETS_TO_RUN, RUN_MODE, ARMS, 'seeds=', SEEDS)
print('Outputs:', ARTIFACT_ROOT)


## 2. Môi trường và snapshot mã nguồn

Clone khi repo chưa tồn tại, giữ nguyên checkout có sẵn. Kiểm tra commit nếu đã chỉ định.
Pip bị ràng buộc theo Torch đang cài để giữ CUDA wheel. Imports được kiểm tra trong subprocess mới.
Không xoá module kernel, không cài lại Torch và không force-reset repo.


In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
required_code = ['main_stair4_v4.py', 'models/stair4_v4.py', 'models/stair4_v4_graph.py',
                 'models/stair4_v4_utils.py', 'optimizers/stair4_v4_smoother.py',
                 'tests/test_stair4_v4.py', 'requirements-stair4-v4.txt']
missing = [p for p in required_code if not (REPO / p).is_file()]
if missing:
    raise FileNotFoundError(f'Upload/push the missing v4 files first: {missing}')
try:
    GIT_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    GIT_STATUS = subprocess.check_output(['git', 'status', '--short'], cwd=REPO, text=True)
except (subprocess.CalledProcessError, FileNotFoundError):
    GIT_SHA, GIT_STATUS = 'unavailable', 'Uploaded source without Git metadata'
if EXPECTED_COMMIT and GIT_SHA != EXPECTED_COMMIT:
    raise RuntimeError(f'Expected {EXPECTED_COMMIT}, found {GIT_SHA}')
print('Commit:', GIT_SHA, '\n', GIT_STATUS)
torch_version_before = importlib.metadata.version('torch')
if INSTALL_DEPENDENCIES:
    constraint = ARTIFACT_ROOT / 'torch-constraint.txt'
    constraint.write_text(f'torch=={torch_version_before}\n', encoding='utf-8')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stair4-v4.txt'),
                    '-c', str(constraint), 'matplotlib', 'pandas'], cwd=REPO, env=ENV, check=True)
if importlib.metadata.version('torch') != torch_version_before:
    raise RuntimeError('The installed PyTorch version changed')
probe = """
import json, platform, importlib.metadata, torch
import models.freerec_compat
from models.stair4_v4 import STAIR4V4
from models.stair4_v4_graph import CSGCOptions
def _safe_ver(n):
    try:
        return importlib.metadata.version(n)
    except Exception:
        return 'not-installed'
r = {'python': platform.python_version(), 'cuda': torch.cuda.is_available(),
     'packages': {n: _safe_ver(n) for n in
                  ('torch', 'freerec', 'torchdata', 'torch-geometric', 'numpy', 'scipy', 'numba')}}
if r['cuda']:
    r['gpu'] = torch.cuda.get_device_name(0)
    r['vram_gib'] = torch.cuda.get_device_properties(0).total_memory / 2**30
print('V4_RUNTIME_JSON=' + json.dumps(r))
"""
checked = subprocess.run([sys.executable, '-c', probe], cwd=REPO, env=ENV,
                         check=True, capture_output=True, text=True, encoding='utf-8')
print(checked.stderr)
RUNTIME = json.loads(next(x for x in checked.stdout.splitlines() if x.startswith('V4_RUNTIME_JSON=')).split('=', 1)[1])
if DEVICE != 'cpu' and not RUNTIME['cuda']:
    raise RuntimeError('Enable a Kaggle GPU accelerator or explicitly set DEVICE="cpu"')
print(json.dumps(RUNTIME, indent=2))
(ARTIFACT_ROOT / 'runtime.json').write_text(json.dumps(RUNTIME, indent=2), encoding='utf-8')
(ARTIFACT_ROOT / 'pip-freeze.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], env=ENV, text=True), encoding='utf-8')


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def code_fingerprints():
    paths = [REPO / 'main.py', REPO / 'main_stair4_v4.py', REPO / 'requirements-stair4-v4.txt',
             *sorted((REPO / 'models').glob('*.py')), *sorted((REPO / 'optimizers').glob('*.py')),
             REPO / 'tests/test_stair4_v4.py', REPO / 'conftest.py', REPO / 'tests/conftest.py',
             REPO / 'scripts/summarize_stair4_runs.py',
             REPO / 'scripts/audit_stair4_v4_graph.py',
             *sorted((REPO / 'configs').glob('dataset_stair4_v4_*.yaml')),
             *sorted((REPO / 'configs').glob('Amazon2014*_550_MMRec.yaml'))]
    paths = [p for p in paths if p.is_file()]
    return {p.relative_to(REPO).as_posix(): file_sha256(p) for p in paths}


In [ ]:
SOURCE_HASHES = code_fingerprints()
source_manifest = ARTIFACT_ROOT / 'source_manifest.json'
if source_manifest.exists():
    previous = json.loads(source_manifest.read_text(encoding='utf-8'))
    if previous['files'] != SOURCE_HASHES:
        raise RuntimeError('Source differs from this experiment. Use a new EXPERIMENT_ID.')
source_manifest.write_text(json.dumps({'git_sha': GIT_SHA, 'git_status': GIT_STATUS,
                                      'files': SOURCE_HASHES}, indent=2), encoding='utf-8')
for relative in SOURCE_HASHES:
    destination = ARTIFACT_ROOT / 'source' / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO / relative, destination)
print('Recorded source files:', len(SOURCE_HASHES))


## 3. Dò và liên kết dữ liệu

Chỉ nhận thư mục có đủ năm file FreeRec; không tự split/reindex hoặc chuyển .inter/.npy.
Nếu có nhiều bản Baby/Sports/Electronics, đặt DATASET_SOURCES chính xác.
Đọc SHA-256 theo block, sau đó liên kết từng file vào một thư mục writable gắn với fingerprint.
Không copy schema.pkl cũ: FreeRec tạo schema mới tại đây. Dataset/feature lớn không đưa vào ZIP output.


In [ ]:
def discover_dataset(key, input_root, explicit=None):
    def complete(path):
        return path.is_dir() and all((path / name).is_file() for name in REQUIRED_FILES)
    if explicit is not None:
        path = Path(explicit).resolve()
        if not complete(path):
            raise FileNotFoundError(f'Incomplete dataset: {path}; required={REQUIRED_FILES}')
        return path
    aliases = {'baby': ('baby',), 'sports': ('sport',), 'electronics': ('electronic',)}[key]
    candidates = set()
    if Path(input_root).is_dir():
        for root, dirs, files in os.walk(input_root, followlinks=False):
            path = Path(root)
            if set(REQUIRED_FILES).issubset(files) and any(a in str(path).lower() for a in aliases):
                candidates.add(path.resolve())
    if len(candidates) != 1:
        raise RuntimeError(f'{key}: found {len(candidates)} complete datasets: {sorted(map(str, candidates))}. '
                           'Set DATASET_SOURCES explicitly; archives must be extracted first.')
    return candidates.pop()

def stage_dataset(source, key, base_root):
    source = Path(source).resolve()
    names = list(REQUIRED_FILES)
    names += [p.name for p in sorted(source.iterdir()) if p.is_file() and p.name not in names
              and (p.name == 'config.yaml' or any(x in p.name.lower() for x in ('mapping', 'id2', '2id')))]
    fingerprint = {name: {'bytes': (source / name).stat().st_size, 'sha256': file_sha256(source / name)}
                   for name in names}
    key_hash = hashlib.sha256(json.dumps(fingerprint, sort_keys=True).encode()).hexdigest()[:16]
    root = Path(base_root) / key_hash
    destination = root / 'Processed' / DATASET_NAMES[key]
    destination.mkdir(parents=True, exist_ok=True)
    for name in names:
        src, dst = source / name, destination / name
        if dst.exists() or dst.is_symlink():
            if dst.resolve() == src.resolve():
                continue
            if dst.is_file() and file_sha256(dst) == fingerprint[name]['sha256']:
                continue
            raise RuntimeError(f'Existing dataset file differs: {dst}')
        try:
            dst.symlink_to(src)
        except OSError:
            if shutil.disk_usage(root).free < src.stat().st_size + 512 * 1024**2:
                raise RuntimeError(f'Insufficient space to copy {src}')
            shutil.copy2(src, dst)
    return {'source': str(source), 'root': str(root), 'processed': str(destination),
            'fingerprint': key_hash, 'files': fingerprint}


In [ ]:
PREPARED = {}
for key in DATASETS_TO_RUN:
    source = discover_dataset(key, INPUT_ROOT, DATASET_SOURCES[key])
    PREPARED[key] = stage_dataset(source, key, DATA_ROOT)
    print(key, '->', PREPARED[key]['processed'])
(ARTIFACT_ROOT / 'data_manifest.json').write_text(json.dumps(PREPARED, indent=2), encoding='utf-8')


## 4. Preflight và cấu hình YAML đã resolve

Tests kiểm tra thống kê, Gate 0 nhiều bước Adam, scoring, checkpoint/resume thật và CUDA khi có GPU.
Mọi import/test failure làm pipeline dừng. Deprecation warning TorchData/CSR được giữ lại.
Batch/lr/weight decay/gamma và tiêu chí NDCG@20 lấy từ YAML kế thừa baseline.


In [ ]:
test_dir = ARTIFACT_ROOT / 'preflight' / uuid.uuid4().hex
test_dir.mkdir(parents=True, exist_ok=True)
preflight = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_stair4_v4.py', '-q',
                            '-p', 'no:cacheprovider', '--basetemp', str(test_dir)],
                           cwd=REPO, env=ENV, capture_output=True, text=True, encoding='utf-8')
(ARTIFACT_ROOT / 'preflight.log').write_text(preflight.stdout + preflight.stderr, encoding='utf-8')
print(preflight.stdout, preflight.stderr)
preflight.check_returncode()
resolve_code = """
import json
from main_stair4_v4 import load_config
r = {k: load_config('configs/dataset_stair4_v4_' + k + '.yaml') for k in ('baby', 'sports', 'electronics')}
print('V4_CONFIG_JSON=' + json.dumps(r))
"""
resolved = subprocess.run([sys.executable, '-c', resolve_code], cwd=REPO, env=ENV,
                          check=True, capture_output=True, text=True, encoding='utf-8')
CONFIGS = json.loads(next(x for x in resolved.stdout.splitlines() if x.startswith('V4_CONFIG_JSON=')).split('=', 1)[1])
for key in DATASETS_TO_RUN:
    print(key, {name: CONFIGS[key][name] for name in ('batch_size', 'lr', 'weight_decay', 'gamma', 'epochs', 'which4best')})
(ARTIFACT_ROOT / 'resolved_configs.json').write_text(json.dumps(CONFIGS, indent=2), encoding='utf-8')


## 5. Runner có log, kiểm tra trạng thái và lưu checkpoint

Mỗi run có thư mục riêng. Rerun cell tái sử dụng run hoàn tất với cùng command/source/data; run lỗi phải resume rõ ràng.
Interrupt sẽ terminate subprocess. Sau mỗi run, best/last model và FreeRec logs được copy vào output.
Chỉ JSONL đo thực tế được dùng; không scrape maxima trong bảng log.


In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    for number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Malformed JSONL at {path}:{number}') from exc
    return rows

def preserve_freerec_outputs(run_dir):
    manifest_file = Path(run_dir) / 'run_manifest.json'
    if not manifest_file.exists():
        return
    manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
    log_path = manifest.get('protocol', {}).get('freerec_log_path')
    if not log_path:
        return
    source = Path(log_path)
    if not source.is_absolute():
        source = REPO / source
    destination = Path(run_dir) / 'freerec'
    if source.is_dir() and source.resolve() != destination.resolve():
        shutil.copytree(source, destination, dirs_exist_ok=True, ignore=shutil.ignore_patterns('checkpoints'))

def run_command(command, run_dir, metadata, resume=False):
    run_dir = Path(run_dir)
    if code_fingerprints() != SOURCE_HASHES:
        raise RuntimeError('Source changed after preflight. Start a new experiment.')
    command_path = run_dir / 'command.json'
    if command_path.exists() and not resume:
        previous = json.loads(command_path.read_text(encoding='utf-8'))
        if previous['argv'] != command or previous['source_files'] != SOURCE_HASHES or previous['metadata'] != metadata:
            raise RuntimeError(f'Run configuration/source/data changed: {run_dir}')
        manifest_path = run_dir / 'run_manifest.json'
        if manifest_path.exists() and json.loads(manifest_path.read_text(encoding='utf-8'))['status'] in {'completed', 'audit_completed'}:
            print('Reusing completed run:', run_dir)
            return run_dir
        raise RuntimeError(f'Incomplete run: {run_dir}. Set RESUME_RUN or use a new experiment ID.')
    run_dir.mkdir(parents=True, exist_ok=True)
    attempt = {'argv': command, 'cwd': str(REPO), 'source_files': SOURCE_HASHES, 'metadata': metadata,
               'started': time.strftime('%Y-%m-%d %H:%M:%S'), 'resume': resume}
    if not resume:
        command_path.write_text(json.dumps(attempt, indent=2), encoding='utf-8')
    with (run_dir / 'attempts.jsonl').open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(attempt) + '\n')
    print('Running:', ' '.join(command))
    process, outcome = None, 'failed'
    started = time.perf_counter()
    try:
        with (run_dir / 'console.log').open('a' if resume else 'w', encoding='utf-8') as log:
            process = subprocess.Popen(command, cwd=REPO, env=ENV, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end='')
            status = process.wait()
            if status:
                raise subprocess.CalledProcessError(status, command)
        manifest = json.loads((run_dir / 'run_manifest.json').read_text(encoding='utf-8'))
        if manifest['status'] not in {'completed', 'audit_completed'}:
            raise RuntimeError(f'Process exited without a completed manifest: {run_dir}')
        outcome = manifest['status']
    except KeyboardInterrupt:
        outcome = 'interrupted'
        raise
    finally:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        if process is not None and process.stdout is not None:
            process.stdout.close()
        preserve_freerec_outputs(run_dir)
        with (run_dir / 'attempt_results.jsonl').open('a', encoding='utf-8') as stream:
            stream.write(json.dumps({'status': outcome, 'seconds': time.perf_counter() - started}) + '\n')
    return run_dir

def make_command(key, arm, seed, stage, epochs):
    run_dir = ARTIFACT_ROOT / key / stage / f'{arm}_seed{seed}'
    config = REPO / 'configs' / f'dataset_stair4_v4_{key}.yaml'
    if not config.is_file():
        raise FileNotFoundError(config)
    command = [sys.executable, '-u', str(REPO / 'main_stair4_v4.py'), '--config', str(config),
               '--root', PREPARED[key]['root'], '--device', DEVICE, '--seed', str(seed),
               '--num-workers', '0', '--epochs', str(epochs), '--ablation-id', arm,
               '--run-dir', str(run_dir), '--checkpoint-dir', str(run_dir / 'checkpoints'),
               '--graph-cache-dir', str(CACHE_ROOT), '--id', f'{EXPERIMENT_ID}_{key}_{stage}_{arm}_{seed}']
    for name, value in PRESETS[arm].items():
        command += ['--' + name.replace('_', '-'), str(value)]
    if stage == 'audit':
        command.append('--audit-only')
    return command, run_dir

def launch(key, arm, seed, stage, epochs):
    command, directory = make_command(key, arm, seed, stage, epochs)
    return run_command(command, directory, {'dataset': key, 'arm': arm, 'seed': seed,
                       'stage': stage, 'data_fingerprint': PREPARED[key]['fingerprint']})


## 6. Graph audit và benchmark ba epochs

Quan sát tỷ lệ cạnh có bằng chứng, shrinkage, dấu trọng số và độ thay đổi toán tử; không suy gain từ graph audit.
Audit còn chạy baseline MI/SVD. Benchmark đo cả preparation và thời gian epoch; bỏ epoch đầu khi xem median steady-state.
Không dùng con số thời gian/VRAM ghi cứng từ v3/v2.


In [ ]:
AUDITS = {}
if RESUME_RUN is None:
    for key in DATASETS_TO_RUN:
        directory = launch(key, 'V4-C', SEEDS[0], 'audit', 1)
        AUDITS[key] = json.loads((directory / 'graph_audit.json').read_text(encoding='utf-8'))
        print(key, json.dumps(AUDITS[key], indent=2))
        if AUDITS[key].get('evidence_fraction', 0) == 0:
            print('No overlap evidence; the core may provide no calibration signal.')
    if RUN_BENCHMARK:
        for key in DATASETS_TO_RUN:
            for arm in ['V4-B1', 'V4-C']:
                launch(key, arm, SEEDS[0], 'benchmark', BENCHMARK_EPOCHS)
else:
    print('Resume mode: fresh audit/benchmark skipped.')


In [ ]:
def train_dataset(key):
    if key not in DATASETS_TO_RUN:
        print('Not selected:', key)
        return
    if RESUME_RUN is not None:
        print('Resume mode: fresh training skipped.')
        return
    epochs = PILOT_EPOCHS if RUN_MODE == 'pilot' else int(CONFIGS[key]['epochs'])
    for seed in SEEDS:
        for arm in ARMS:
            launch(key, arm, seed, RUN_MODE, epochs)
    # Tự động hiển thị VRAM và Learning Curves ngay khi hoàn tất
    plot_vram_profile(key)
    plot_single_dataset_learning_curves(key)

DATASET_PROFILES = {
    'baby': {'name': 'Amazon Baby', 'color': '#ff7f0e'},
    'sports': {'name': 'Amazon Sports', 'color': '#1f77b4'},
    'electronics': {'name': 'Amazon Electronics', 'color': '#2ca02c'}
}
ARM_COLORS = {
    'V4-B1': '#1f77b4',
    'V4-C': '#ff7f0e',
    'V4-NS': '#2ca02c',
    'V4-ND': '#d62728',
    'V4-NA': '#9467bd',
    'V4-SH': '#8c564b',
    'V4-SM': '#e377c2',
}
BASELINE_REF = {
    'sports': {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'baby': {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0427, 'Recall@20': 0.0665, 'NDCG@10': 0.0233, 'NDCG@20': 0.0294}
}

def plot_vram_profile(key, dataset_name=None, output_filename=None):
    """Trực quan hóa mức tiêu thụ bộ nhớ GPU (PyTorch Peak Allocated VRAM - Paper Standard)."""
    import matplotlib.pyplot as plt
    info = DATASET_PROFILES.get(key, {'name': key.capitalize(), 'color': '#ff7f0e'})
    disp_name = dataset_name if dataset_name else info['name']

    target_dirs = []
    base_dir = ARTIFACT_ROOT / key
    if base_dir.is_dir():
        for stage in [RUN_MODE, 'benchmark', 'audit']:
            s_dir = base_dir / stage
            if s_dir.is_dir():
                for p in sorted(s_dir.iterdir()):
                    if p.is_dir() and (p / 'epochs.jsonl').exists():
                        target_dirs.append(p)
            if target_dirs and stage == RUN_MODE:
                break

    fig, ax = plt.subplots(figsize=(10, 4.5), dpi=150)
    has_data = False
    all_peaks = []

    for run_dir in target_dirs:
        epochs_data = read_jsonl(run_dir / 'epochs.jsonl')
        if not epochs_data:
            continue
        epochs_data = sorted(epochs_data, key=lambda x: x['epoch'])
        seen = set()
        clean_epochs = []
        for r in reversed(epochs_data):
            if r['epoch'] not in seen:
                seen.add(r['epoch'])
                clean_epochs.append(r)
        clean_epochs.reverse()

        ep_nums = [r['epoch'] for r in clean_epochs]
        vram_vals = [r.get('peak_allocated_mb', 0) for r in clean_epochs]
        if not vram_vals or max(vram_vals) == 0:
            continue

        has_data = True
        arm_name = run_dir.name
        color = ARM_COLORS.get(arm_name.split('_')[0], info['color'])
        peak = max(vram_vals)
        all_peaks.append(peak)

        ax.plot(ep_nums, vram_vals, color=color, lw=2.0, label=f'{arm_name} (Peak: {peak:.1f} MiB)', zorder=4)
        ax.fill_between(ep_nums, vram_vals, color=color, alpha=0.12, zorder=3)

    if has_data:
        max_peak = max(all_peaks)
        ax.axhline(max_peak, color='#d62728', linestyle='--', lw=1.3,
                   label=f'Global Measured Peak: {max_peak:.1f} MiB ({max_peak/1024:.2f} GiB)', zorder=5)
        ax.set_title(f'Model Tensor Memory Profile (PyTorch Peak Allocated) — {disp_name}', fontsize=12.5, fontweight='bold', pad=12)
        ax.set_xlabel('Training Epoch', fontsize=10.5, labelpad=8)
        ax.set_ylabel('Peak Allocated Memory (MiB)', fontsize=10.5)
        ax.grid(True, linestyle='--', alpha=0.35)
        ax.legend(loc='lower right', fontsize=8.5)
    else:
        ax.text(0.5, 0.5, f"Chưa có dữ liệu telemetry VRAM thực tế cho {disp_name}\n(Thực thi cell huấn luyện bên trên để ghi nhận)",
                ha='center', va='center', transform=ax.transAxes, color='#777777', fontsize=11)
        ax.set_title(f'Model Tensor Memory Profile — {disp_name} [Đang chờ thực thi]', fontsize=12.5, fontweight='bold')
        ax.set_xlabel('Training Epoch', fontsize=10.5)
        ax.set_ylabel('Peak Allocated Memory (MiB)', fontsize=10.5)
        ax.grid(True, linestyle='--', alpha=0.35)

    plt.tight_layout()
    if not output_filename:
        output_filename = REPORT_DIR / f'vram_profile_{key}.png'
    output_filename = Path(output_filename)
    output_filename.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✅ [VRAM Profile Saved] -> {output_filename}')

def plot_single_dataset_learning_curves(key, dataset_name=None, output_filename=None):
    """Đồ thị 4 panel chuẩn mực công bố khoa học theo dõi động lực học STAIR4-CSGC."""
    import matplotlib.pyplot as plt
    info = DATASET_PROFILES.get(key, {'name': key.capitalize(), 'color': '#ff7f0e'})
    disp_name = dataset_name if dataset_name else info['name']

    target_dirs = []
    base_dir = ARTIFACT_ROOT / key
    if base_dir.is_dir():
        for stage in [RUN_MODE, 'benchmark']:
            s_dir = base_dir / stage
            if s_dir.is_dir():
                for p in sorted(s_dir.iterdir()):
                    if p.is_dir() and (p / 'epochs.jsonl').exists():
                        target_dirs.append(p)
            if target_dirs and stage == RUN_MODE:
                break

    fig, axes = plt.subplots(1, 4, figsize=(22, 4.8), dpi=150)
    has_data = False

    for run_dir in target_dirs:
        epochs_data = read_jsonl(run_dir / 'epochs.jsonl')
        eval_data = read_jsonl(run_dir / 'evaluation.jsonl')
        if not epochs_data:
            continue
        has_data = True

        seen = set()
        clean_epochs = []
        for r in reversed(sorted(epochs_data, key=lambda x: x['epoch'])):
            if r['epoch'] not in seen:
                seen.add(r['epoch'])
                clean_epochs.append(r)
        clean_epochs.reverse()

        ep_nums = [r['epoch'] for r in clean_epochs]
        bpr_losses = [r.get('bpr', 0) for r in clean_epochs]
        train_secs = [r.get('train_seconds', 0) for r in clean_epochs]

        arm_name = run_dir.name
        color = ARM_COLORS.get(arm_name.split('_')[0], info['color'])

        axes[0].plot(ep_nums, bpr_losses, color=color, lw=1.8, label=f'{arm_name}')

        valid_records = [r for r in eval_data if r.get('mode') == 'valid' and not r.get('selected_checkpoint', False)]
        if valid_records:
            valid_dict = {}
            for r in valid_records:
                valid_dict[r['epoch']] = r['metrics']
            v_epochs = sorted(valid_dict.keys())
            ndcgs = [valid_dict[ep].get('NDCG@20', 0) for ep in v_epochs]
            recalls = [valid_dict[ep].get('Recall@20', 0) for ep in v_epochs]

            axes[1].plot(v_epochs, ndcgs, color=color, lw=2.0, label=f'{arm_name}')
            if ndcgs:
                best_n_idx = int(np.argmax(ndcgs))
                best_n_ep, best_n_val = v_epochs[best_n_idx], ndcgs[best_n_idx]
                axes[1].axvline(best_n_ep, color=color, linestyle=':', lw=1.2, alpha=0.7)
                axes[1].scatter([best_n_ep], [best_n_val], color=color, s=40, zorder=5)

            axes[2].plot(v_epochs, recalls, color=color, lw=2.0, label=f'{arm_name}')
            if recalls:
                best_r_idx = int(np.argmax(recalls))
                best_r_ep, best_r_val = v_epochs[best_r_idx], recalls[best_r_idx]
                axes[2].axvline(best_r_ep, color=color, linestyle=':', lw=1.2, alpha=0.7)
                axes[2].scatter([best_r_ep], [best_r_val], color=color, s=40, zorder=5)

        if len(train_secs) > 1:
            axes[3].plot(ep_nums[1:], train_secs[1:], color=color, lw=1.6, label=f'{arm_name}')
        else:
            axes[3].plot(ep_nums, train_secs, color=color, lw=1.6, label=f'{arm_name}')

    ref = BASELINE_REF.get(key, {})
    if has_data:
        title_suffix = f' ({RUN_MODE.upper()})'
        if 'NDCG@20' in ref:
            axes[1].axhline(ref['NDCG@20'], color='#777777', linestyle='--', lw=1.2,
                            label=f'STAIR Base: {ref["NDCG@20"]:.4f}', alpha=0.85)
        if 'Recall@20' in ref:
            axes[2].axhline(ref['Recall@20'], color='#777777', linestyle='--', lw=1.2,
                            label=f'STAIR Base: {ref["Recall@20"]:.4f}', alpha=0.85)

        fig.suptitle(f'STAIR4-CSGC Training and Validation Dynamics — {disp_name}{title_suffix}',
                     fontsize=14, fontweight='bold', y=0.98)
        for ax in axes:
            ax.grid(True, linestyle='--', alpha=0.35)
            if ax.lines:
                ax.legend(loc='best', fontsize=8.5)
    else:
        fig.suptitle(f'STAIR4-CSGC Training and Validation Dynamics — {disp_name} [Đang Chờ Huấn Luyện]',
                     fontsize=14, fontweight='bold', y=0.98)
        for ax in axes:
            ax.text(0.5, 0.5, "Chưa có dữ liệu huấn luyện\n(Thực thi cell huấn luyện để ghi nhận)",
                    ha='center', va='center', transform=ax.transAxes, color='#777777', fontsize=10.5)
            ax.grid(True, linestyle='--', alpha=0.35)

    axes[0].set_title('(a) Training Loss Trajectory (BPR)', fontweight='bold', fontsize=11.5)
    axes[0].set_xlabel('Epoch', fontsize=10)
    axes[0].set_ylabel('BPR Loss', fontsize=10)

    axes[1].set_title('(b) Validation NDCG@20', fontweight='bold', fontsize=11.5)
    axes[1].set_xlabel('Epoch', fontsize=10)
    axes[1].set_ylabel('NDCG@20', fontsize=10)

    axes[2].set_title('(c) Validation Recall@20', fontweight='bold', fontsize=11.5)
    axes[2].set_xlabel('Epoch', fontsize=10)
    axes[2].set_ylabel('Recall@20', fontsize=10)

    axes[3].set_title('(d) Computation Time (s/epoch)', fontweight='bold', fontsize=11.5)
    axes[3].set_xlabel('Epoch', fontsize=10)
    axes[3].set_ylabel('Seconds / Epoch', fontsize=10)

    plt.tight_layout(rect=[0, 0.045, 1, 0.96])
    if not output_filename:
        output_filename = REPORT_DIR / f'learning_curve_{key}.png'
    output_filename = Path(output_filename)
    output_filename.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✅ [Learning Curves Saved] -> {output_filename}')

## 7. Training — Amazon Baby


In [ ]:
train_dataset('baby')


## Cell 7b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Baby (Paper Standard)
Trực quan hóa mức tiêu thụ VRAM thuần bộ nhớ tensor (`torch.cuda.max_memory_allocated`) trong suốt quá trình huấn luyện Amazon Baby.

In [ ]:
# Cell 7b: Model Tensor VRAM Profile — Amazon Baby (Paper Standard)
plot_vram_profile('baby')

## Cell 7c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Baby (Paper Standard)
Hiển thị 4 panel chuẩn mực công bố: Training Loss Trajectory, Validation NDCG@20, Validation Recall@20 và Computation Time.

In [ ]:
# Cell 7c: Learning Dynamics & Convergence Profiles — Amazon Baby (Paper Standard)
plot_single_dataset_learning_curves('baby')

## 8. Training — Amazon Sports


In [ ]:
train_dataset('sports')


## Cell 8b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Sports (Paper Standard)
Trực quan hóa mức tiêu thụ VRAM thuần bộ nhớ tensor (`torch.cuda.max_memory_allocated`) trong suốt quá trình huấn luyện Amazon Sports.

In [ ]:
# Cell 8b: Model Tensor VRAM Profile — Amazon Sports (Paper Standard)
plot_vram_profile('sports')

## Cell 8c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Sports (Paper Standard)
Hiển thị 4 panel chuẩn mực công bố: Training Loss Trajectory, Validation NDCG@20, Validation Recall@20 và Computation Time.

In [ ]:
# Cell 8c: Learning Dynamics & Convergence Profiles — Amazon Sports (Paper Standard)
plot_single_dataset_learning_curves('sports')

## 9. Training — Amazon Electronics


In [ ]:
train_dataset('electronics')


## Cell 9b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Electronics (Paper Standard)
Trực quan hóa mức tiêu thụ VRAM thuần bộ nhớ tensor (`torch.cuda.max_memory_allocated`) an toàn trên catalog khổng lồ 63K items.

In [ ]:
# Cell 9b: Model Tensor VRAM Profile — Amazon Electronics (Paper Standard)
plot_vram_profile('electronics')

## Cell 9c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Electronics (Paper Standard)
Hiển thị 4 panel chuẩn mực công bố: Training Loss Trajectory, Validation NDCG@20, Validation Recall@20 và Computation Time.

In [ ]:
# Cell 9c: Learning Dynamics & Convergence Profiles — Amazon Electronics (Paper Standard)
plot_single_dataset_learning_curves('electronics')

## 10. Resume


In [ ]:
def resume_training(run_dir, target_epochs=None):
    run_dir = Path(run_dir).resolve()
    saved = json.loads((run_dir / 'command.json').read_text(encoding='utf-8'))
    if saved['source_files'] != SOURCE_HASHES:
        raise RuntimeError('Resume source fingerprint differs; restore the saved source snapshot.')
    metadata = saved['metadata']
    if metadata['stage'] == 'audit':
        raise ValueError('An audit has no training checkpoint')
    key = metadata['dataset']
    if key not in PREPARED or metadata['data_fingerprint'] != PREPARED[key]['fingerprint']:
        raise RuntimeError('Resume dataset differs or is not selected')
    command = saved['argv'].copy()
    if not (run_dir / 'checkpoints' / 'checkpoint.tar').is_file():
        raise FileNotFoundError('No checkpoint.tar; start a new run if the first checkpoint was not saved')
    if Path(command[command.index('--run-dir') + 1]).resolve() != run_dir:
        raise ValueError('Restore the bundle to its original run path')
    command[0] = sys.executable
    command[command.index('--root') + 1] = PREPARED[key]['root']
    if target_epochs is not None:
        if int(target_epochs) < int(command[command.index('--epochs') + 1]):
            raise ValueError('Resume target cannot be smaller than the original target')
        command[command.index('--epochs') + 1] = str(int(target_epochs))
    command.append('--resume')
    return run_command(command, run_dir, metadata, resume=True)


In [ ]:
if RESUME_RUN is not None:
    resume_training(RESUME_RUN, RESUME_TARGET_EPOCHS)
else:
    print('No resume requested.')


## 11. Metrics tại checkpoint được chọn


In [ ]:
METRICS = ['RECALL@10', 'RECALL@20', 'NDCG@10', 'NDCG@20']

def collect_results(root):
    import pandas as pd
    rows, manifests = [], {}
    for path in sorted(Path(root).glob('*/*/*/run_manifest.json')):
        directory = path.parent
        m = json.loads(path.read_text(encoding='utf-8'))
        command_file = directory / 'command.json'
        if not command_file.exists():
            continue
        saved = json.loads(command_file.read_text(encoding='utf-8'))
        meta = saved['metadata']
        if meta['stage'] == 'audit':
            continue
        row = {'run': str(directory), 'dataset': meta['dataset'], 'stage': meta['stage'],
               'arm': m['options']['ablation_id'], 'seed': m['seed'], 'status': m['status'],
               'selected_epoch': m.get('selected_epoch'), 'preparation_seconds': m.get('preparation_seconds')}
        attempts = read_jsonl(directory / 'attempt_results.jsonl')
        row['all_attempt_seconds'] = sum(a['seconds'] for a in attempts) if attempts else m.get('total_seconds')
        epochs = {r['epoch']: r for r in read_jsonl(directory / 'epochs.jsonl')}
        elapsed = [r['train_seconds'] for epoch, r in sorted(epochs.items()) if epoch > 1]
        row['median_train_seconds_after_epoch1'] = float(pd.Series(elapsed, dtype=float).median()) if elapsed else float('nan')
        device = saved['argv'][saved['argv'].index('--device') + 1]
        row['peak_allocated_mb'] = max((r.get('peak_allocated_mb', 0) for r in epochs.values()), default=float('nan')) if device != 'cpu' else float('nan')
        for mode in ('valid', 'test'):
            for metric in METRICS:
                row[f'{mode}_{metric}'] = float('nan')
        if m['status'] == 'completed':
            for record in read_jsonl(directory / 'evaluation.jsonl'):
                if record.get('selected_checkpoint') and record['epoch'] == m.get('selected_epoch'):
                    for metric in METRICS:
                        if metric in record['metrics']:
                            row[f"{record['mode']}_{metric}"] = record['metrics'][metric]
        rows.append(row)
        manifests[str(directory)] = m
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    for metric in ('RECALL@20', 'NDCG@20'):
        frame[f'delta_test_{metric}_pct'] = float('nan')
    for index, row in frame.iterrows():
        controls = frame[(frame.dataset == row.dataset) & (frame.stage == row.stage) &
                         (frame.seed == row.seed) & (frame.arm == 'V4-B1') & (frame.status == 'completed')]
        if len(controls) != 1 or row.status != 'completed':
            continue
        baseline = controls.iloc[0]
        current, reference = manifests[row.run], manifests[baseline.run]
        same_data = all(current['graph_identity'].get(k) == reference['graph_identity'].get(k)
                        for k in ('raw', 'baseline', 'train_crow', 'train_col', 'train_shape', 'data'))
        same_protocol = all(current['protocol'].get(k) == reference['protocol'].get(k)
                            for k in ('ranking', 'selection', 'lr', 'weight_decay', 'batch_size', 'epochs', 'gamma', 'num_workers'))
        if not (same_data and same_protocol and current['source_hash'] == reference['source_hash']
                and current['runtime'] == reference['runtime']):
            continue
        for metric in ('RECALL@20', 'NDCG@20'):
            value, base = row[f'test_{metric}'], baseline[f'test_{metric}']
            if pd.notna(value) and pd.notna(base) and base > 0:
                frame.loc[index, f'delta_test_{metric}_pct'] = 100 * (value / base - 1)
    return frame


In [ ]:
import pandas as pd
RESULTS = collect_results(ARTIFACT_ROOT)
if RESULTS.empty:
    print('No training runs found.')
else:
    RESULTS.to_csv(REPORT_DIR / 'all_runs.csv', index=False)
    PRIMARY = RESULTS[RESULTS.stage.isin(['pilot', 'full'])].copy()
    display(PRIMARY)
    display(RESULTS[RESULTS.stage == 'benchmark'][['dataset', 'arm', 'seed', 'status',
            'preparation_seconds', 'median_train_seconds_after_epoch1', 'peak_allocated_mb']])
    PRIMARY.to_csv(REPORT_DIR / 'selected_checkpoint_metrics.csv', index=False)
    completed = PRIMARY[PRIMARY.status == 'completed']
    if not completed.empty:
        aggregate = completed.groupby(['dataset', 'stage', 'arm'])[[f'test_{m}' for m in METRICS]].agg(['mean', 'std', 'count'])
        aggregate.to_csv(REPORT_DIR / 'seed_summary.csv')
        display(aggregate)  # One seed yields NaN std, not evidence of significance.
    columns = ['dataset', 'stage', 'arm', 'seed', 'selected_epoch'] + [f'test_{m}' for m in METRICS]
    (REPORT_DIR / 'selected_metrics.tex').write_text(PRIMARY[columns].to_latex(index=False,
        na_rep='--', float_format=lambda x: f'{x:.5f}', escape=True,
        caption='Test metrics at the validation-selected checkpoint. Pilot and full runs are separate.'), encoding='utf-8')


## 12. Tổng hợp Đa Tập Dữ Liệu: Learning curves, thời gian và peak memory


In [ ]:
def plot_runs(results, output):
    import matplotlib.pyplot as plt
    import pandas as pd
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    if results.empty:
        return []
    files = []
    for (dataset, stage), group in results.groupby(['dataset', 'stage']):
        if stage not in ('pilot', 'full'):
            continue
        fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
        memory_fig, memory_ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
        for row in group.itertuples():
            label = f'{row.arm} / seed {row.seed}'
            train = pd.DataFrame(read_jsonl(Path(row.run) / 'epochs.jsonl'))
            if not train.empty:
                train = train.drop_duplicates('epoch', keep='last').sort_values('epoch')
                axes[0, 0].plot(train.epoch, train.bpr, label=label)
                axes[1, 1].plot(train.epoch, train.train_seconds, label=label)
                if pd.notna(row.peak_allocated_mb):
                    memory_ax.plot(train.epoch, train.peak_allocated_mb, label=label)
            valid = [r for r in read_jsonl(Path(row.run) / 'evaluation.jsonl')
                     if r['mode'] == 'valid' and not r.get('selected_checkpoint', False)]
            valid = sorted({r['epoch']: r for r in valid}.values(), key=lambda r: r['epoch'])
            for axis, metric in ((axes[0, 1], 'NDCG@20'), (axes[1, 0], 'Recall@20')):
                observed = [(r['epoch'], r['metrics'].get(metric, r['metrics'].get(metric.upper()))) for r in valid if metric in r['metrics'] or metric.upper() in r['metrics']]
                if observed:
                    x, y = zip(*observed)
                    axis.plot(x, y, label=label)
        for axis, title in zip(axes.flat, ['Training BPR', 'Validation NDCG@20', 'Validation Recall@20', 'Training seconds per epoch']):
            axis.set(title=title, xlabel='Epoch')
            axis.grid(alpha=.25)
            if axis.lines:
                axis.legend(fontsize=8)
        fig.suptitle(f'STAIR4-CSGC | {dataset} | {stage}')
        path = output / f'learning_{dataset}_{stage}.png'
        fig.savefig(path, dpi=180)
        files.append(path)
        plt.show()
        plt.close(fig)
        if memory_ax.lines:
            memory_ax.set(title=f'{dataset}: cumulative PyTorch peak allocated memory', xlabel='Epoch', ylabel='MiB')
            memory_ax.legend(fontsize=8)
            memory_ax.grid(alpha=.25)
            path = output / f'peak_memory_{dataset}_{stage}.png'
            memory_fig.savefig(path, dpi=180)
            files.append(path)
            plt.show()
        plt.close(memory_fig)
    return files

In [ ]:
FIGURES = plot_runs(RESULTS, REPORT_DIR)
print('Saved figures:', [str(path) for path in FIGURES])


## 13. Đóng gói để tải về


In [ ]:
import zipfile
if MAKE_ARCHIVE:
    archive = ARTIFACT_ROOT.parent / (EXPERIMENT_ID + '_artifacts.zip')
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as bundle:
        for path in sorted(ARTIFACT_ROOT.rglob('*')):
            relative = path.relative_to(ARTIFACT_ROOT)
            if path.is_file() and relative.parts[0] != 'preflight':
                bundle.write(path, Path(EXPERIMENT_ID) / relative)
    print('Archive:', archive, f'({archive.stat().st_size / 2**20:.1f} MiB)')
else:
    print('Artifacts retained:', ARTIFACT_ROOT)


## 14. Đọc kết quả và xử lý lỗi
